# M02A · Text Files in Python & Biological Applications
**Programming for Computational Biology (Python)** · Author: Prateek Paul · Last updated: 2024-10-29

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/meprateekpaul/ProgrammingCB/blob/main/modules/M02_File_Handling/M02A_Text_Files.ipynb)

⏱ ~90 min · Difficulty: ★☆☆ · Prerequisites: Module 1 (Core Python, Variables, Strings, Lists, Loops)

---


## 1. Learning Outcomes

By the end of this notebook, you will be able to:
1. **Navigate** file systems cross-platform using `pathlib.Path` and configure portable path constants (`DATA_DIR`, `OUT_DIR`).
2. **Compare** file opening modes (`r`, `w`, `a`, `x`, `r+`, `b`) and manage file resources deterministically using context managers (`with open(...)`).
3. **Evaluate** memory and execution trade-offs between `.read()`, `.readlines()`, and line-by-line streaming (`for line in f`) for genomic datasets of varying scales.
4. **Safely write and append** formatted sequence data and metadata to text files without unintended data loss.
5. **Clean and parse** raw biological text files by stripping whitespace, skipping headers and comments, and extracting tab-delimited records.
6. **Build** robust file handling operations using explicit defensive error handling (`try / except FileNotFoundError`).

---


## 2. Prerequisites

Before starting this notebook, ensure you are comfortable with:
- Core Python syntax: variables, strings, string slicing (`seq[0:10]`), lists, dictionaries, conditionals (`if/elif/else`), for-loops, and functions (`def`).
- Basic molecular biology concepts: DNA nucleotides (A, C, G, T), GC content percentage, and gene coordinates.
- If you need a refresher, revisit **Module 1 (Core Python & Sequence Manipulation)**.

---


In [ ]:
# --- Setup -------------------------------------------------------
import sys, os
from pathlib import Path

# Detect whether we are running in Google Colab or locally
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !git clone -q https://github.com/meprateekpaul/ProgrammingCB.git
    ROOT = Path("ProgrammingCB")
else:
    # Resolve repository root whether run from repo root, modules/, or solutions/
    ROOT = Path.cwd().resolve()
    while not (ROOT / "data").exists() and ROOT.parent != ROOT:
        ROOT = ROOT.parent

DATA_DIR = ROOT / "data"
OUT_DIR  = ROOT / "outputs" / "M02A"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Python interpreter : {sys.version.split()[0]}")
print(f"Repository ROOT    : {ROOT.resolve()}")
print(f"DATA_DIR exists    : {DATA_DIR.exists()} ({DATA_DIR})")
print(f"OUT_DIR ready      : {OUT_DIR.exists()} ({OUT_DIR})")


## 4. Concept Sections

### 4.1 Concept 1: File Paths & Cross-Platform Navigation with `pathlib`

In bioinformatics, datasets rarely live in the same directory as your Python scripts. You might have raw FASTQ files stored on a shared network drive, intermediate alignment files in a scratch directory, and final summary tables in an output directory.

#### Absolute vs Relative Paths
- An **absolute path** specifies the full location from the root of the file system:
  - macOS/Linux: `/home/user/project/data/sequences/dna_sequences.txt`
  - Windows: `C:\Users\user\project\data\sequences\dna_sequences.txt`
- A **relative path** specifies the location relative to your current working directory:
  - `data/sequences/dna_sequences.txt`

#### Why `pathlib.Path`?
Historically, Python programmers used `os.path.join("data", "sequences", "dna_sequences.txt")` or string concatenation. Python 3 introduced `pathlib`, which provides an object-oriented, cross-platform interface:
- Uses the `/` operator to join paths cleanly on macOS, Linux, and Windows.
- Eliminates Windows backslash bugs (like `\n` or `\t` being parsed as escape characters).
- Provides intuitive methods like `.exists()`, `.is_file()`, `.is_dir()`, `.stat().st_size`, `.name`, and `.parent`.


In [ ]:
# Worked Example 4.1: Exploring paths with pathlib.Path
raw_seq_path = DATA_DIR / "sequences" / "dna_sequences.txt"

print(f"Path object          : {raw_seq_path}")
print(f"File exists?         : {raw_seq_path.exists()}")
print(f"Is a regular file?   : {raw_seq_path.is_file()}")
print(f"File name            : {raw_seq_path.name}")
print(f"Parent directory     : {raw_seq_path.parent.name}")
print(f"File extension       : {raw_seq_path.suffix}")
print(f"Size in bytes        : {raw_seq_path.stat().st_size:,} bytes")


#### ✏️ Try it 4.1: Validate Input File
Check that `dna_sequences.txt` exists and calculate its file size in kilobytes (KB, where $1\text{ KB} = 1024\text{ bytes}$).


In [ ]:
# Try it 4.1: Compute file size in KB
target_file = DATA_DIR / "sequences" / "dna_sequences.txt"

# Calculate size in kilobytes
file_size_kb = target_file.stat().st_size / 1024.0
print(f"File size: {file_size_kb:.2f} KB")

# Test check
assert target_file.exists(), "Target file must exist!"
assert file_size_kb > 100.0, "Expected file size > 100 KB"
print("Try it 4.1 passed ✔")


#### ⚠️ Common Mistakes with File Paths
- **Hardcoding user paths**: Writing `/Users/yourname/data/...` guarantees your code will fail when shared with colleagues, run on an HPC cluster, or opened in Google Colab. Always use relative anchors like `DATA_DIR = ROOT / "data"`.
- **String concatenation with backslashes**: Writing `"data\" + filename` breaks on Linux/macOS and introduces accidental escape sequences (e.g. `	`, `
`). Use `Path / filename`.
- **Assuming current working directory (CWD)**: When launching Jupyter from different folders, `Path.cwd()` may point to different locations. Anchoring relative to the repository `ROOT` avoids broken paths.


### 4.2 Concept 2: Opening Files, Modes, and Context Managers

To work with a file in Python, you open a connection known as a **file handle** or stream using the built-in `open()` function:

```python
file_handle = open(filepath, mode="r", encoding="utf-8")
```

#### File Opening Modes
| Mode | Name | Description | If File Exists | If File Doesn't Exist |
|---|---|---|---|---|
| `'r'` | Read (default) | Opens file for reading text | Starts at beginning | **Raises `FileNotFoundError`** |
| `'w'` | Write | Opens file for writing text | **Truncates (ERASES) file!** | Creates new file |
| `'a'` | Append | Opens file for adding text | Starts at end (preserves data) | Creates new file |
| `'x'` | Exclusive Creation | Creates new file for writing | **Raises `FileExistsError`** | Creates new file |
| `'r+'`| Read + Write | Opens for both reading & writing | Preserves content | Raises `FileNotFoundError` |
| `'b'` | Binary mode | Appended to mode (e.g. `'rb'`, `'wb'`) | Reads/writes raw bytes | Depends on mode |

#### The `with` Statement (Context Manager)
Always use the `with` statement when opening files:
```python
with open(filepath, "r", encoding="utf-8") as f:
    content = f.read()
# File is automatically and guaranteed to be closed here!
```
Even if an unhandled error or exception occurs inside the `with` block, Python **guarantees** the file handle is cleanly closed, flushing buffered data to disk and releasing operating system resources.


In [ ]:
# Worked Example 4.2: Writing with a context manager and verifying closure
test_file = OUT_DIR / "primers_test.txt"

# 1. Write PCR primers to file
with open(test_file, mode="w", encoding="utf-8") as f:
    f.write("# Forward and Reverse primers for TP53 exon 4\n")
    f.write("TP53_ex4_F\t5'-TGACTGTTACCAGACCCA-3'\n")
    f.write("TP53_ex4_R\t5'-AGGGGAGTGCAAAGATGT-3'\n")

# Verify that the file handle is automatically closed outside the with block
print(f"File handle closed: {f.closed}")

# 2. Read back the file
with open(test_file, mode="r", encoding="utf-8") as f:
    lines = f.readlines()

print(f"Read {len(lines)} lines successfully:")
for line in lines:
    print("  ", line.strip())


#### ✏️ Try it 4.2: Safe Writing & Verification
Write a small file `sample_ids.txt` to `OUT_DIR` containing 3 sample identifiers (`"SMP_001"`, `"SMP_002"`, `"SMP_003"`), each on a new line. Then open and read the file to confirm it contains 3 lines.


In [ ]:
# Try it 4.2: Write and verify sample identifiers
sample_file = OUT_DIR / "sample_ids.txt"
samples = ["SMP_001", "SMP_002", "SMP_003"]

with open(sample_file, mode="w", encoding="utf-8") as f:
    for s in samples:
        f.write(f"{s}\n")

with open(sample_file, mode="r", encoding="utf-8") as f:
    saved_samples = [line.strip() for line in f if line.strip()]

assert saved_samples == samples, f"Expected {samples}, got {saved_samples}"
assert sample_file.exists(), "File should exist"
print("Try it 4.2 passed ✔")


#### ⚠️ Common Mistakes with Opening Files
- **The Accidental Truncation Bug (`'w'`)**: Opening an existing dataset with `mode='w'` immediately zeroes out the file on disk. If you wanted to add data, you must use `mode='a'`.
- **Forgetting `encoding='utf-8'`**: Different operating systems default to different encodings (e.g., `cp1252` on Windows, `UTF-8` on Linux/macOS). Omitting `encoding='utf-8'` causes mysterious decoding errors when reading Greek symbols (e.g. $\alpha$, $\beta$) or degree signs in thermal cycler protocols.
- **Resource Leaks without `with`**: Using `f = open(...)` without `f.close()` holds open OS file descriptors. In loops processing thousands of FASTQ files, this causes `OSError: [Errno 24] Too many open files`.


### 4.3 Concept 3: Three Ways to Read Files (Memory & Scale in Biology)

Biological files range from tiny metadata tables to massive gigabyte-scale sequencing archives. Choosing the wrong reading method will crash your environment.

| Method | Syntax | Memory Footprint | Output Type | When to Use |
|---|---|---|---|---|
| **Whole File** | `f.read()` | **$O(N)$** — Loads entire file into RAM at once | Single `str` | Small files (<50 MB), regex matching spanning line breaks |
| **All Lines** | `f.readlines()` | **$O(N)$** — Loads entire file into a list in RAM | `list[str]` | Medium files when you need random index access (e.g. `lines[42]`) |
| **Line-by-Line Streaming** | `for line in f:` | **$O(1)$** — Only one line in memory at any instant | Iterator yielding `str` | **Default choice for genomics!** Essential for large FASTQ/SAM/VCF files |

#### The Golden Rule of Bioinformatics File Handling:
> *If a file could reasonably exceed a few hundred megabytes, NEVER use `.read()` or `.readlines()`. Stream it line-by-line using `for line in f:`.*


In [ ]:
# Worked Example 4.3: Streaming vs readlines
seq_file = DATA_DIR / "sequences" / "dna_sequences.txt"

# 1. Memory-efficient streaming: count lines and find headers
header_count = 0
sequence_lines = 0

with open(seq_file, "r", encoding="utf-8") as f:
    for line in f:
        if line.startswith(">"):
            header_count += 1
        elif line.strip():
            sequence_lines += 1

print(f"Streaming result: {header_count} headers, {sequence_lines} sequence lines.")


#### ✏️ Try it 4.3: Streaming Line Counter
Count the total number of lines in `dna_sequences.txt` using a memory-efficient `for line in f:` streaming loop.


In [ ]:
# Try it 4.3: Streaming line count
total_lines = 0
with open(DATA_DIR / "sequences" / "dna_sequences.txt", "r", encoding="utf-8") as f:
    for line in f:
        total_lines += 1

print(f"Total lines counted: {total_lines}")
assert total_lines > 1000, "Expected over 1000 lines in dna_sequences.txt"
print("Try it 4.3 passed ✔")


#### ⚠️ Common Mistakes when Reading Files
- **Calling `.read()` on sequencing data**: Attempting to read a 40 GB FASTQ file into RAM will instantly trigger Python's `MemoryError` or cause the OS kernel to kill your process.
- **Reading from an exhausted handle**: Once you iterate through `for line in f:` to the end, the internal file pointer is at the end of the file. A second loop `for line in f:` will yield zero iterations unless you rewind the pointer using `f.seek(0)`.
- **Forgetting newline characters in lines**: Both `f.readlines()` and `for line in f:` include the trailing newline (`\n` or `\r\n`). Failing to call `.strip()` or `.rstrip('\n')` leads to incorrect string lengths and failed comparisons.


### 4.4 Concept 4: Writing and Appending Files

Writing output files is fundamental to reproducible pipelines. Python provides three primary ways to output text:
1. `f.write(text_string)`: Writes the exact string provided. **Does NOT append a newline automatically!**
2. `f.writelines(iterable_of_strings)`: Writes each string in a sequence. **Also does NOT add newlines!**
3. `print(..., file=f)`: Uses Python's `print` function to write to the file handle, automatically adding `\n` by default.

#### Appending vs Overwriting
- Mode `'w'`: Always creates a new file or truncates an existing file to 0 bytes before writing.
- Mode `'a'`: Preserves existing data and places the write pointer at the end of the file.


In [ ]:
# Worked Example 4.4: Writing and appending formatted sequence statistics
log_file = OUT_DIR / "analysis_run.log"

# Step 1: Initialize log file with run metadata (mode 'w')
with open(log_file, "w", encoding="utf-8") as f:
    f.write("=== Bioinformatics Pipeline Log ===\n")
    f.write("Status: INITIALIZED\n")

# Step 2: Simulate adding experiment steps over time (mode 'a')
steps = [
    "Step 1: Quality filtering completed - 100% reads passed",
    "Step 2: GC content calculated - Mean GC: 52.4%",
    "Step 3: Output exported to FASTA"
]

with open(log_file, "a", encoding="utf-8") as f:
    for step in steps:
        print(step, file=f)  # print automatically handles the trailing newline!

# Inspect final log
with open(log_file, "r", encoding="utf-8") as f:
    print(f.read())


#### ✏️ Try it 4.4: Appending to an Audit Trail
Append an audit entry `"Pipeline completed successfully"` to `log_file` using `'a'` mode, and verify that the file now contains 6 lines.


In [ ]:
# Try it 4.4: Append audit entry
with open(log_file, "a", encoding="utf-8") as f:
    f.write("Status: Pipeline completed successfully\n")

with open(log_file, "r", encoding="utf-8") as f:
    lines = [l.strip() for l in f if l.strip()]

assert len(lines) == 6, f"Expected 6 non-empty lines, got {len(lines)}"
assert "Status: Pipeline completed successfully" in lines[-1]
print("Try it 4.4 passed ✔")


#### ⚠️ Common Mistakes when Writing Files
- **Forgetting trailing newlines with `.write()`**: Calling `f.write("ATGC")` followed by `f.write("GCTA")` writes `"ATGCGCTA"` on one line.
- **Passing non-strings to `.write()`**: Calling `f.write(42)` or `f.write(["A", "C"])` raises `TypeError: write() argument must be str`. You must convert explicitly: `f.write(str(42) + "\n")`.
- **Assuming data is immediately on disk**: Operating systems buffer file writes. Data is only guaranteed to be written to disk when the buffer is flushed or the file is closed via the `with` block.


### 4.5 Concept 5: Cleaning Lines & Parsing Tabular Biological Data

Biological files frequently arrive with unwanted formatting:
- Windows CRLF line endings (`\r\n`) vs Unix LF (`\n`).
- Leading/trailing whitespace or tabs.
- Comment lines starting with `#` or metadata headers.
- Blank lines inserted between records.

#### String Cleaning Tools
- `.strip()`: Removes all leading and trailing whitespace, tabs, and newlines.
- `.rstrip('\r\n')`: Removes trailing newline characters while preserving any intentional leading indentation.
- `.split('\t')`: Splits a tab-separated line into a list of column values.


In [ ]:
# Worked Example 4.5: Parsing a messy TSV annotation table
raw_tsv_data = """# Gene Annotation Table: Build GRCh38
# gene_id\tchrom\tstart\tend\tstrand\tbiotype
ENSG00000141510\tchr17\t7668402\t7687550\t-\tprotein_coding
ENSG00000012048\tchr17\t43044295\t43170245\t-\tprotein_coding

ENSG00000139618\tchr13\t32315086\t32400266\t+\tprotein_coding
# End of file
"""

# Parse the text into structured records
parsed_genes = []
for raw_line in raw_tsv_data.splitlines():
    line = raw_line.strip()
    # Skip empty lines and comment lines
    if not line or line.startswith("#"):
        continue
    
    parts = line.split("\t")
    gene_id, chrom, start, end, strand, biotype = parts
    length = int(end) - int(start) + 1
    
    parsed_genes.append({
        "gene_id": gene_id,
        "chrom": chrom,
        "length_bp": length,
        "strand": strand
    })

print(f"Extracted {len(parsed_genes)} genes:")
for g in parsed_genes:
    print(f"  {g['gene_id']} on {g['chrom']} ({g['strand']}) length: {g['length_bp']} bp")


#### ✏️ Try it 4.5: Clean Raw Sequences
Given a list of messy sequence lines containing whitespace, lowercase letters, and empty lines, clean them into uppercase strings and filter out empty items.


In [ ]:
# Try it 4.5: Clean messy sequence strings
messy_inputs = ["  atgcgtacgt\n", "  \t\n", "ggccaatt  \n", ""]
cleaned = [line.strip().upper() for line in messy_inputs if line.strip()]

assert cleaned == ["ATGCGTACGT", "GGCCAATT"], f"Got: {cleaned}"
print("Try it 4.5 passed ✔")


#### ⚠️ Common Mistakes with Parsing
- **Using `.split(" ")` on TSV files**: If columns are separated by tabs (`\t`), splitting by space will fail to split columns. Always inspect the delimiter.
- **Forgetting to cast numeric strings**: File readers return strings. If you extract coordinates `start = "100"`, `start + 50` will raise `TypeError: can only concatenate str to str`. Cast with `int(start)`.
- **IndexError on blank lines**: Splitting an unstripped blank line like `""` yields `[""]`. Attempting `parts[1]` immediately raises `IndexError: list index out of range`. Always check `if not line: continue` first.


### 4.6 Concept 6: Defensive Programming & Error Handling

Biological workflows frequently run non-interactively in automated pipelines overnight. If a single sample file is missing or corrupted, an unhandled exception will terminate the entire pipeline, wasting hours of compute time.

#### Catching `FileNotFoundError`
Instead of letting Python crash, wrap file-opening logic in a `try / except` block:
```python
try:
    with open(path, "r", encoding="utf-8") as f:
        data = f.read()
except FileNotFoundError:
    print(f"Warning: File {path} was not found. Skipping sample.")
    data = None
```


In [ ]:
# Worked Example 4.6: Safe file reader with validation
def safe_read_first_line(filepath: Path) -> str | None:
    """Safely read the first line of a file, returning None if missing."""
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            for line in f:
                cleaned = line.strip()
                if cleaned:
                    return cleaned
            return ""  # File was empty or only whitespace
    except FileNotFoundError:
        return None
    except PermissionError:
        print(f"Error: Insufficient permissions to read {filepath}")
        return None

# Test on valid file
existing_first = safe_read_first_line(DATA_DIR / "sequences" / "dna_sequences.txt")
print("First line of existing file :", existing_first[:60], "...")

# Test on nonexistent file
missing_result = safe_read_first_line(DATA_DIR / "sequences" / "nonexistent_file.txt")
print("Result on nonexistent file  :", missing_result)


#### ✏️ Try it 4.6: Safe File Existence Check
Write an assertion confirming that calling `safe_read_first_line` on a bogus path returns `None` without raising an exception.


In [ ]:
# Try it 4.6: Verify safe file handling
result = safe_read_first_line(OUT_DIR / "definitely_missing_file_123.txt")
assert result is None, "Missing file should return None safely"
print("Try it 4.6 passed ✔")


#### ⚠️ Common Mistakes in Error Handling
- **Bare `except:` clauses**: Writing `except:` catches EVERYTHING, including `KeyboardInterrupt` (Ctrl+C) and Python `SyntaxError`. Always catch specific exceptions: `except FileNotFoundError:`.
- **Silent failure without logging**: Swallowing an error without returning `None`, printing a log, or raising a custom error leaves you wondering why your downstream dataframe has zero rows.


---
## 5. Practice Exercises

Apply your knowledge to solve the following bioinformatics file handling problems.
- **Level 1 (★)**: Direct application of a single concept.
- **Level 2 (★★)**: Integrating 2–3 concepts (parsing, cleaning, filtering).
- **Level 3 (★★★)**: Advanced real-world genomic streaming and reporting.

---


### Ex 2A.1 (★) — Count Non-Empty Lines in a File

**Problem Statement:**  
Write a function `count_non_empty_lines(filepath: Path) -> int` that opens the given file and returns the total number of lines that contain non-whitespace characters.

- **Input:** `filepath: Path` pointing to a text file.
- **Output:** `int` representing count of non-empty lines.


In [ ]:
def count_non_empty_lines(filepath: Path) -> int:
    # TODO: your code here
    ...


In [ ]:
# ✅ Test Ex 2A.1
test_path = OUT_DIR / "test_ex2a1.txt"
with open(test_path, "w", encoding="utf-8") as f:
    f.write("Line 1\n\n   \nLine 2\nLine 3\n\n")

assert count_non_empty_lines(test_path) == 3, "Should count exactly 3 non-empty lines"
assert count_non_empty_lines(DATA_DIR / "sequences" / "dna_sequences.txt") > 1000
print("Ex 2A.1 passed ✔")


### Ex 2A.2 (★) — Clean and Filter Raw Sequence Lines

**Problem Statement:**  
Write a function `clean_and_write_sequences(raw_path: Path, clean_path: Path) -> int` that:
1. Reads `raw_path` line by line.
2. Strips all leading/trailing whitespace.
3. Skips lines that are empty or start with `#` (comment lines).
4. Converts all remaining sequence lines to uppercase.
5. Writes the cleaned sequence lines to `clean_path` (each on its own line).
6. Returns the number of cleaned sequence lines written.

- **Input:** `raw_path: Path` (input file), `clean_path: Path` (output destination).
- **Output:** `int` count of cleaned lines written.


In [ ]:
def clean_and_write_sequences(raw_path: Path, clean_path: Path) -> int:
    # TODO: your code here
    ...


In [ ]:
# ✅ Test Ex 2A.2
raw_test = OUT_DIR / "raw_test_ex2a2.txt"
clean_test = OUT_DIR / "clean_test_ex2a2.txt"

with open(raw_test, "w", encoding="utf-8") as f:
    f.write("# Sample Raw Reads\n  atgcgt  \n\n# Low quality\n  tgca  \n")

written_count = clean_and_write_sequences(raw_test, clean_test)
assert written_count == 2, f"Expected 2 lines written, got {written_count}"

with open(clean_test, "r", encoding="utf-8") as f:
    lines = [l.strip() for l in f]

assert lines == ["ATGCGT", "TGCA"], f"Expected ['ATGCGT', 'TGCA'], got {lines}"
print("Ex 2A.2 passed ✔")


### Ex 2A.3 (★★) — Extract Specific Columns from a TSV File

**Problem Statement:**  
Write a function `extract_tsv_columns(tsv_path: Path, col_indices: list[int]) -> list[tuple[str, ...]]` that:
1. Reads a tab-delimited file.
2. Skips comment lines starting with `#` and empty lines.
3. For each data row, extracts the columns specified by `col_indices` (0-based indexing) and stores them as a tuple of strings.
4. Returns a list of these extracted tuples.

- **Input:** `tsv_path: Path`, `col_indices: list[int]` (e.g. `[0, 2]`).
- **Output:** `list[tuple[str, ...]]`.


In [ ]:
def extract_tsv_columns(tsv_path: Path, col_indices: list[int]) -> list[tuple[str, ...]]:
    # TODO: your code here
    ...


In [ ]:
# ✅ Test Ex 2A.3
tsv_sample = OUT_DIR / "sample_data_ex2a3.tsv"
with open(tsv_sample, "w", encoding="utf-8") as f:
    f.write("# Gene\tOrganism\tLength\n")
    f.write("TP53\tHuman\t2512\n")
    f.write("lacZ\tE. coli\t3075\n")
    f.write("\n")
    f.write("INS\tHuman\t468\n")

res = extract_tsv_columns(tsv_sample, [0, 2])
assert res == [("TP53", "2512"), ("lacZ", "3075"), ("INS", "468")], f"Got: {res}"
print("Ex 2A.3 passed ✔")


### Ex 2A.4 (★★) — Merge Two Files with a Delimiter

**Problem Statement:**  
Write a function `merge_text_files(file1_path: Path, file2_path: Path, out_path: Path, delimiter: str = "---") -> int` that:
1. Copies all content from `file1_path` into `out_path`.
2. Inserts a delimiter line (e.g. `"---"`).
3. Appends all content from `file2_path` into `out_path`.
4. Returns the total number of lines in `out_path`.

- **Input:** Paths to two input files, an output path, and an optional delimiter string.
- **Output:** `int` total line count of the resulting merged file.


In [ ]:
def merge_text_files(file1_path: Path, file2_path: Path, out_path: Path, delimiter: str = "---") -> int:
    # TODO: your code here
    ...


In [ ]:
# ✅ Test Ex 2A.4
f1 = OUT_DIR / "merge_f1.txt"
f2 = OUT_DIR / "merge_f2.txt"
f_merged = OUT_DIR / "merged_result.txt"

with open(f1, "w", encoding="utf-8") as f:
    f.write("Batch A: Sample 1\nBatch A: Sample 2\n")
with open(f2, "w", encoding="utf-8") as f:
    f.write("Batch B: Sample 3\nBatch B: Sample 4\n")

total_count = merge_text_files(f1, f2, f_merged, delimiter="### BATCH BREAK ###")
assert f_merged.exists()
with open(f_merged, "r", encoding="utf-8") as f:
    merged_text = f.read()

assert "Batch A: Sample 1" in merged_text
assert "### BATCH BREAK ###" in merged_text
assert "Batch B: Sample 4" in merged_text
print("Ex 2A.4 passed ✔")


### Ex 2A.5 (★★) — Reverse File Line Order

**Problem Statement:**  
Write a function `reverse_file_contents(in_path: Path, out_path: Path) -> None` that reads a file and writes its lines in reverse order (the last line of `in_path` becomes the first line of `out_path`, etc.).

- **Input:** `in_path: Path`, `out_path: Path`.
- **Output:** None (writes reversed lines to disk).


In [ ]:
def reverse_file_contents(in_path: Path, out_path: Path) -> None:
    # TODO: your code here
    ...


In [ ]:
# ✅ Test Ex 2A.5
rev_in = OUT_DIR / "rev_in.txt"
rev_out = OUT_DIR / "rev_out.txt"

with open(rev_in, "w", encoding="utf-8") as f:
    f.write("Line 1\nLine 2\nLine 3\n")

reverse_file_contents(rev_in, rev_out)
with open(rev_out, "r", encoding="utf-8") as f:
    rev_lines = [l.strip() for l in f if l.strip()]

assert rev_lines == ["Line 3", "Line 2", "Line 1"], f"Got: {rev_lines}"
print("Ex 2A.5 passed ✔")


### Ex 2A.6 (★★★) — Parse Metadata Table into Typed Dictionaries

**Problem Statement:**  
Write a function `parse_metadata_table(table_path: Path) -> list[dict[str, str | int | float]]` that:
1. Parses a delimited file (handling either tab `\t` or comma `,` delimiters).
2. Skips comment lines (starting with `#`) and empty lines.
3. Uses the first non-comment row as column header names.
4. Parses subsequent rows into dictionaries mapping header name to cell value.
5. Converts numeric values automatically:
   - Integers to `int` (e.g. `"42"` $\rightarrow$ `42`)
   - Floats to `float` (e.g. `"3.14"` $\rightarrow$ `3.14`)
   - Other values remain `str`.

- **Input:** `table_path: Path`.
- **Output:** `list[dict]` containing typed record dictionaries.


In [ ]:
def parse_metadata_table(table_path: Path) -> list[dict[str, str | int | float]]:
    # TODO: your code here
    ...


In [ ]:
# ✅ Test Ex 2A.6
sample_csv = OUT_DIR / "samples_meta.csv"
with open(sample_csv, "w", encoding="utf-8") as f:
    f.write("# Expression Metadata\nsample_id,concentration,replicate,passed_qc\n")
    f.write("S1,14.5,1,true\n")
    f.write("S2,28.0,2,true\n")
    f.write("S3,5.2,3,false\n")

records = parse_metadata_table(sample_csv)
assert len(records) == 3, f"Expected 3 records, got {len(records)}"
assert records[0]["sample_id"] == "S1"
assert records[0]["concentration"] == 14.5
assert isinstance(records[0]["concentration"], float)
assert records[0]["replicate"] == 1
assert isinstance(records[0]["replicate"], int)
assert records[2]["passed_qc"] == "false"
print("Ex 2A.6 passed ✔")


### Ex 2A.7 (★★★) — Stream Biological Sequences and Compute Summary Statistics

**Problem Statement:**  
Write a function `stream_fasta_summary_stats(txt_path: Path, out_summary_path: Path) -> dict[str, float | int]` that:
1. Streams `txt_path` line by line without loading the full file into memory.
2. Identifies headers (lines starting with `>`) and sequence lines.
3. Computes:
   - `total_sequences`: number of headers found.
   - `total_nucleotides`: total character count of all sequence lines (ignoring whitespace).
   - `gc_count`: total count of 'G' and 'C' bases (case-insensitive).
   - `overall_gc_pct`: $(gc\_count / total\_nucleotides) \times 100$, rounded to 2 decimal places (or `0.0` if no nucleotides).
4. Writes a formatted summary text file to `out_summary_path`.
5. Returns a dictionary with keys: `{"total_sequences": int, "total_nucleotides": int, "overall_gc_pct": float}`.

- **Input:** `txt_path: Path`, `out_summary_path: Path`.
- **Output:** `dict` of summary statistics.


In [ ]:
def stream_fasta_summary_stats(txt_path: Path, out_summary_path: Path) -> dict[str, float | int]:
    # TODO: your code here
    ...


In [ ]:
# ✅ Test Ex 2A.7
stats_out = OUT_DIR / "dna_sequences_summary.txt"
stats = stream_fasta_summary_stats(DATA_DIR / "sequences" / "dna_sequences.txt", stats_out)

assert stats["total_sequences"] >= 1, "Should find at least 1 sequence header"
assert stats["total_nucleotides"] > 10000, "Should have over 10,000 nucleotides"
assert 0.0 <= stats["overall_gc_pct"] <= 100.0, "GC% must be between 0 and 100"
assert stats_out.exists(), "Output summary file must be created"
print("Stats computed:", stats)
print("Ex 2A.7 passed ✔")


---
## 6. Summary & Cheat Sheet

| Task | Syntax | Memory Complexity | Notes |
|---|---|---|---|
| Join Paths | `DATA_DIR / "sequences" / "file.txt"` | $O(1)$ | Cross-platform, handles slashes automatically |
| Path Check | `path.exists()`, `path.is_file()` | $O(1)$ | Safe verification before opening |
| Open for Reading | `with open(p, "r", encoding="utf-8") as f:` | — | Always specify encoding; auto-closes file |
| Open for Writing | `with open(p, "w", encoding="utf-8") as f:` | — | **Warning:** Overwrites/truncates existing file! |
| Open for Appending | `with open(p, "a", encoding="utf-8") as f:` | — | Preserves content and writes to end |
| Stream Lines | `for line in f:` | **$O(1)$** | **Essential for bioinformatics scale** |
| Read All Lines | `lines = f.readlines()` | $O(N)$ | Loads entire file into a list of strings |
| Read Full File | `text = f.read()` | $O(N)$ | Loads entire file into a single string |
| Strip Whitespace | `line.strip()`, `line.rstrip('\r\n')` | $O(L)$ | Cleans newline characters and tabs |
| Split Columns | `line.split('\t')` | $O(L)$ | Standard delimiter parsing for GFF, BED, TSV |
| Defensive Catch | `try: ... except FileNotFoundError:` | — | Prevents catastrophic crashes in automated pipelines |

---


## 7. Further Reading & Resources

- [Python Official Documentation: `pathlib` — Object-oriented filesystem paths](https://docs.python.org/3/library/pathlib.html)
- [Real Python: Reading and Writing Files in Python (Guide)](https://realpython.com/read-write-files-python/)
- [Real Python: Python's `with` Statement and Context Managers](https://realpython.com/python-with-statement/)
- [Biopython Tutorial and Cookbook](https://biopython.org/DIST/docs/tutorial/Tutorial.html)
- [UCSB Bioinformatics: Working with Large Genomic Files](https://eriqande.github.io/rep-res-eeb-2017/unix-and-bash.html)
